# 05 — Hyperparameter Tuning (KAGGLE GPU)
## WavSent-MTL · Tasks 2.1–2.9

**Runs on: Kaggle T4 x2 GPU**

**What this notebook does:**
- Random search: 40 trials per model (TKAN, LSTM, GRU, TCN)
- Metric: minimize val_loss on Kotekar dataset
- Done ONCE — same best params applied to Kaggle dataset
- Saves: `best_params_{model}.json` for each model

**After this notebook:**
1. Download all 4 `best_params_*.json` files
2. Manually update `config/config.py` `best_params` section
3. Push updated config to GitHub

In [1]:
# ── Kaggle Setup ────────────────────────────────────────────────────────────
import subprocess
subprocess.run(['git', 'clone', 'https://github.com/IAMNEERAJ05/WavSent-MTL.git',
                '/kaggle/working/WavSent-MTL'], check=True)

import sys
import os
sys.path.insert(0, '/kaggle/working/WavSent-MTL')
os.chdir('/kaggle/working/WavSent-MTL')

import torch
from config.config import CONFIG

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__}')
print(f'Device:  {device}')
if device == 'cuda':
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
print('CONFIG loaded.')

Cloning into '/kaggle/working/WavSent-MTL'...


PyTorch: 2.10.0+cu128
Device:  cuda
GPU:     Tesla T4
CONFIG loaded.


## Load Kotekar Processed Arrays

In [2]:
import numpy as np
import json

# Kaggle dataset path — upload data/processed/kotekar/ as 'wavsent-kotekar-processed'
KOTEKAR_INPUT = '/kaggle/input/datasets/consistency23/wavsent-kotekar-processed'

data = {
    'X_train':      np.load(f'{KOTEKAR_INPUT}/X_train.npy'),
    'X_val':        np.load(f'{KOTEKAR_INPUT}/X_val.npy'),
    'X_test':       np.load(f'{KOTEKAR_INPUT}/X_test.npy'),
    'y_clf_train':  np.load(f'{KOTEKAR_INPUT}/y_clf_train.npy'),
    'y_clf_val':    np.load(f'{KOTEKAR_INPUT}/y_clf_val.npy'),
    'y_clf_test':   np.load(f'{KOTEKAR_INPUT}/y_clf_test.npy'),
    'y_reg_train':  np.load(f'{KOTEKAR_INPUT}/y_reg_train.npy'),
    'y_reg_val':    np.load(f'{KOTEKAR_INPUT}/y_reg_val.npy'),
    'y_reg_test':   np.load(f'{KOTEKAR_INPUT}/y_reg_test.npy'),
}

with open(f'{KOTEKAR_INPUT}/selected_features.json') as f:
    selected_features = json.load(f)

N_FEATURES = data['X_train'].shape[2]   # 8
print(f'X_train:          {data["X_train"].shape}')
print(f'X_val:            {data["X_val"].shape}')
print(f'n_features:       {N_FEATURES}')
print(f'selected_features:{selected_features}')

X_train:          (741, 5, 8)
X_val:            (155, 5, 8)
n_features:       8
selected_features:['WILLIAMS_R', 'STOCH_K', 'MACD', 'RSI_14', 'CCI_20', 'EMA_9', 'Volume_d', 'polarity_mean']


## Tasks 2.3–2.6 — Random Search (40 trials × 4 models)

> Each model takes ~5-10 min on T4 GPU. Total ~30-40 min.

In [3]:
from src.training.hyperparam_tuning import random_search

best_params_all = {}

for model_name in CONFIG['pso_models']:   # ['tkan', 'lstm', 'gru', 'tcn']
    print(f'\n{"=" * 60}')
    print(f'Random search: {model_name.upper()}  ({CONFIG["n_search_trials"]} trials)')
    print(f'{"=" * 60}')

    best = random_search(
        model_name=model_name,
        n_features=N_FEATURES,
        data=data,
        n_trials=CONFIG['n_search_trials'],
        device=device,
    )

    best_params_all[model_name] = best

    # Save immediately after each model
    out_path = f'/kaggle/working/best_params_{model_name}.json'
    with open(out_path, 'w') as f:
        json.dump(best, f, indent=2)
    print(f'Saved: {out_path}')

    # Clear GPU memory between models
    import gc
    torch.cuda.empty_cache()
    gc.collect()


Random search: TKAN  (40 trials)


/kaggle/working/WavSent-MTL/src/models/mtl_model.py:114: UserWarning: tkan package not found. Using LSTMEncoder as fallback. Install: pip install git+https://github.com/remigenet/TKAN.git
  encoder = TKANEncoder(


[tkan] Trial 1/40 | val_loss=0.718036 | params={'hidden_size': 64, 'dropout': 0.1, 'learning_rate': 0.0005, 'batch_size': 64}


/kaggle/working/WavSent-MTL/src/models/mtl_model.py:114: UserWarning: tkan package not found. Using LSTMEncoder as fallback. Install: pip install git+https://github.com/remigenet/TKAN.git
  encoder = TKANEncoder(


[tkan] Trial 2/40 | val_loss=0.520958 | params={'hidden_size': 64, 'dropout': 0.2, 'learning_rate': 0.001, 'batch_size': 32}
[tkan] Trial 3/40 | val_loss=0.521579 | params={'hidden_size': 32, 'dropout': 0.3, 'learning_rate': 0.001, 'batch_size': 32}
[tkan] Trial 4/40 | val_loss=0.550919 | params={'hidden_size': 32, 'dropout': 0.1, 'learning_rate': 0.001, 'batch_size': 32}
[tkan] Trial 5/40 | val_loss=0.790425 | params={'hidden_size': 32, 'dropout': 0.3, 'learning_rate': 0.0001, 'batch_size': 16}
[tkan] Trial 6/40 | val_loss=0.631675 | params={'hidden_size': 32, 'dropout': 0.2, 'learning_rate': 0.001, 'batch_size': 64}
[tkan] Trial 7/40 | val_loss=0.816220 | params={'hidden_size': 128, 'dropout': 0.2, 'learning_rate': 0.0001, 'batch_size': 32}
[tkan] Trial 8/40 | val_loss=0.674027 | params={'hidden_size': 128, 'dropout': 0.1, 'learning_rate': 0.0005, 'batch_size': 32}
[tkan] Trial 9/40 | val_loss=0.759764 | params={'hidden_size': 64, 'dropout': 0.1, 'learning_rate': 0.0005, 'batch_size'

## Summary of Best Params

In [4]:
import pandas as pd

print('=' * 60)
print('Best hyperparameters per model:')
print('=' * 60)
for model_name, params in best_params_all.items():
    print(f'\n{model_name.upper()}:')
    for k, v in params.items():
        print(f'  {k}: {v}')

print()
print('Files saved to /kaggle/working/:')
for model_name in best_params_all:
    print(f'  best_params_{model_name}.json')

print()
print('NEXT STEPS:')
print('1. Download all 4 best_params_*.json files')
print('2. Update config/config.py best_params section manually')
print('3. Push config to GitHub')
print('4. Run notebook 06_training_kotekar')

Best hyperparameters per model:

TKAN:
  hidden_size: 64
  dropout: 0.1
  learning_rate: 0.001
  batch_size: 32

LSTM:
  hidden_size: 64
  dropout: 0.1
  learning_rate: 0.001
  batch_size: 32
  num_layers: 2

GRU:
  hidden_size: 128
  dropout: 0.3
  learning_rate: 0.001
  batch_size: 32
  num_layers: 1

TCN:
  hidden_size: 64
  dropout: 0.1
  learning_rate: 0.001
  batch_size: 32
  kernel_size: 2
  num_levels: 2

Files saved to /kaggle/working/:
  best_params_tkan.json
  best_params_lstm.json
  best_params_gru.json
  best_params_tcn.json

NEXT STEPS:
1. Download all 4 best_params_*.json files
2. Update config/config.py best_params section manually
3. Push config to GitHub
4. Run notebook 06_training_kotekar
